In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dev.academy;
CREATE VOLUME dev.academy.flatfiles_managed;

In [0]:
# --- Parámetros de configuración ---
catalog = "dev"
schema = "academy"
vol_int = "flatfiles_managed"
country = "USA"
yyyy, mm, dd = "2025", "10", "13"
# --- Construcción de la ruta base ---
base_int_path = f"/Volumes/{catalog}/{schema}/{vol_int}"

print(f"La ruta base del volumen es: {base_int_path}")

In [0]:
import os
# --- Ruta completa con la estructura estándar ---
landing_path = f"{base_int_path}/country={country}/yyyy={yyyy}/mm={mm}/dd={dd}"
print(f"Se creará la siguiente estructura: {landing_path}")
# --- Creación de los directorios ---
os.makedirs(landing_path, exist_ok=True)

In [0]:
import urllib.request
src_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"

# --- Nombre del archivo de destino ---
file_name = "yellow_tripdata_2024-01.parquet"
dst_parquet_path = f"{landing_path}/{file_name}"

if not os.path.exists(dst_parquet_path):
    print(f"Descargando archivo desde: {src_url}")
    urllib.request.urlretrieve(src_url, dst_parquet_path)
    print(f"Archivo guardado en: {dst_parquet_path}")
else:
    print("El archivo ya existe en la ubicación de destino.")

In [0]:
# Lee todos los archivos .parquet de la ruta de destino y los carga en un DataFrame
df_int = spark.read.parquet(f"{landing_path}/*.parquet")
print(f"Se cargaron {df_int.count()} filas en el DataFrame.")
display(df_int.limit(5)) 

In [0]:

table_name = "taxinyc_internal"
full_table_name = f"{catalog}.{schema}.{table_name}"

# Guarda el DataFrame como una Delta Table
df_int.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(full_table_name)

print(f"El DataFrame se ha guardado como la Delta Table: '{full_table_name}'")


In [0]:
%sql
SELECT * FROM dev.academy.taxinyc_internal LIMIT 10;